In [1]:
import importlib
import pypath.inputs.corum as corum

importlib.reload(corum)  # editable install olduğu için değişiklikleri yansıtır

# 1) İnsan komplekslerini çek
complexes_human = corum.corum_complexes(organism=9606)
print(f"Human complexes: {len(complexes_human)}")

# 2) Filtresiz (tüm organizmalar) çekip toplamla, atlanan satır sayısını kıyaslayalım
complexes_all = corum.corum_complexes(organism=None)
print(f"All organisms (filtered function still drops unrecognized organism strings like 'Mammalia'): {len(complexes_all)}")

# 3) Birkaç örneğe yakından bak
sample_keys = list(complexes_human.keys())[:5]
for k in sample_keys:
    c = complexes_human[k]
    print("-" * 60)
    print("name:", c.name)
    print("ids:", c.ids)
    print("components (uniprot):", list(c.components.keys()))
    print("gene_names:", c.attrs.get('gene_names'))
    print("go_ids:", c.attrs.get('go_ids'))
    print("go_evidences:", c.attrs.get('go_evidences'))
    print("references:", c.references)

# 4) Hizalama sağlaması: her komplekste subunit sayısı = gene_name sayısı mı?
mismatches = [
    k for k, c in complexes_human.items()
    if len(c.components) != len(c.attrs.get('gene_names', []))
]
print(f"\nHizalama uyuşmazlığı olan kompleks sayısı: {len(mismatches)}")
if mismatches:
    print("Örnek:", mismatches[0], complexes_human[mismatches[0]].attrs)

# 5) GO üçlüsü hizalaması (evi/pmid/go_id aynı uzunlukta mı?)
go_mismatches = [
    k for k, c in complexes_human.items()
    if not (len(c.attrs.get('go_evidences', [])) == len(c.attrs.get('go_pubmeds', [])) == len(c.attrs.get('go_ids', [])))
]
print(f"GO üçlü hizalama uyuşmazlığı olan kompleks sayısı: {len(go_mismatches)}")

# 6) "Mammalia" gibi tanınmayan organizma stringleri gerçekten atlanıyor mu?
# (Bunu görmek için ham veriyi ayrı bir kez daha çekip organism kolonunun
# benzersiz değerlerine bakalım)
import csv
import pypath.share.curl as curl
import pypath.resources.urls as urls

c_raw = curl.Curl(urls.urls['corum']['url'], large=True, silent=False)
raw_tab = list(csv.DictReader(c_raw.result, delimiter='\t'))
organisms_seen = set(rec['organism'] for rec in raw_tab)
print(f"\nHam dosyadaki benzersiz organism değerleri: {organisms_seen}")
print(f"Toplam satır sayısı (ham): {len(raw_tab)}")

[2026-08-02 13:58:09] [curl] Module `pysftp` not available. Only downloading of a small number of resources relies on this module. Please install by PIP if it is necessary for you.
Human complexes: 5616
All organisms (filtered function still drops unrecognized organism strings like 'Mammalia'): 8097
------------------------------------------------------------
name: BCL6-HDAC4 complex
ids: defaultdict(<class 'set'>, {'CORUM': {'1'}})
components (uniprot): ['P41182', 'P56524']
gene_names: ['BCL6', 'HDAC4']
go_ids: ['GO:0004407', 'GO:0005634', 'GO:0006265', 'GO:0030183', 'GO:0045892', 'GO:0051276']
go_evidences: ['lit', 'exp', 'lit', 'lit', 'exp', 'lit']
references: {'11929873'}
------------------------------------------------------------
name: BCL6-HDAC5 complex
ids: defaultdict(<class 'set'>, {'CORUM': {'2'}})
components (uniprot): ['P41182', 'Q9UQL6']
gene_names: ['BCL6', 'HDAC5']
go_ids: ['GO:0004407', 'GO:0005634', 'GO:0006265', 'GO:0030183', 'GO:0045892', 'GO:0051276']
go_evidences:

In [9]:
importlib.reload(corum)

complexes_human = corum.corum_complexes(organism=9606)
print(f"Human complexes: {len(complexes_human)}")

sample_keys = list(complexes_human.keys())[:5]
for k in sample_keys:
    c = complexes_human[k]
    print("-" * 60)
    print("name:", c.name)
    print("ids:", c.ids)
    print("components:", list(c.components.keys()))
    print("gene_names:", c.attrs.get('gene_names'))
    print("go_annotations:", c.attrs.get('go_annotations'))
    print("references:", c.references)

# Hizalama artık yapı gereği garanti (tuple set), ama yine de kontrol edelim:
# her go_annotations elemanının 3'lü tuple olduğunu doğrula
bad = [k for k, c in complexes_human.items()
       if any(len(t) != 3 for t in c.attrs.get('go_annotations', set()))]
print(f"\nBozuk go_annotations kaydı: {len(bad)}")

# gene_names artık dict, her uniprot ID component'te var mı kontrol
missing_gene = [k for k, c in complexes_human.items()
                if set(c.components.keys()) - set(c.attrs.get('gene_names', {}).keys())]
print(f"gene_names eksik olan kompleks: {len(missing_gene)}")

Human complexes: 5616
------------------------------------------------------------
name: BCL6-HDAC4 complex
ids: defaultdict(<class 'set'>, {'CORUM': {'1'}})
components: ['P41182', 'P56524']
gene_names: {'P41182': 'BCL6', 'P56524': 'HDAC4'}
go_annotations: {('GO:0051276', 'lit', '11929873'), ('GO:0004407', 'lit', '11929873'), ('GO:0045892', 'exp', '11929873'), ('GO:0006265', 'lit', '11929873'), ('GO:0030183', 'lit', '11929873'), ('GO:0005634', 'exp', '11929873')}
references: {'11929873'}
------------------------------------------------------------
name: BCL6-HDAC5 complex
ids: defaultdict(<class 'set'>, {'CORUM': {'2'}})
components: ['P41182', 'Q9UQL6']
gene_names: {'P41182': 'BCL6', 'Q9UQL6': 'HDAC5'}
go_annotations: {('GO:0051276', 'lit', '11929873'), ('GO:0004407', 'lit', '11929873'), ('GO:0045892', 'exp', '11929873'), ('GO:0006265', 'lit', '11929873'), ('GO:0030183', 'lit', '11929873'), ('GO:0005634', 'exp', '11929873')}
references: {'11929873'}
------------------------------------

In [ ]:
import pypath.utils.taxonomy as taxonomy
import collections

organism_status = collections.Counter()
for rec in raw_tab:
    try:
        tax_id = taxonomy.ensure_ncbi_tax_id(rec['organism'])
        status = f"OK -> {tax_id}" if tax_id else "FAILED (None)"
    except Exception as e:
        status = f"FAILED ({type(e).__name__}: {e})"
    organism_status[(rec['organism'], status)] += 1

for (org, status), count in sorted(organism_status.items()):
    print(f"{org:12s} {status:30s} {count}") 

Bovine       OK -> 9913                     103
Dog          OK -> 9615                     27
Hamster      OK -> 10036                    19
Human        OK -> 9606                     5638
MINK         FAILED (None)                  4
Mammalia     OK -> 40674                    285
Mouse        OK -> 10090                    1333
Pig          OK -> 9823                     13
Rabbit       OK -> 9986                     22
Rat          OK -> 10116                    688


In [6]:
from pypath.utils.taxonomy import get_db
common_db = get_db('common')
print('mink' in common_db, 'Mink' in common_db, 'MINK' in common_db)

False False False


In [7]:
from pypath.utils.taxonomy import get_db
common_db = get_db('common')

print("toplam kayıt sayısı:", len(common_db))

# 'mink' geçen herhangi bir anahtar var mı (kısmi eşleşme)
matches = [k for k in common_db if 'mink' in k.lower()]
print("mink içeren anahtarlar:", matches)

# bilimsel isimle de bir bakalım (Amerikan vizonunun bilimsel adı)
matches2 = [k for k in common_db if 'vison' in k.lower() or 'mustela' in k.lower()]
print("vison/mustela içeren anahtarlar:", matches2)

toplam kayıt sayısı: 34062
mink içeren anahtarlar: ['American mink', 'Bluemink', 'Common minke whale', 'North Pacific minke whale', 'Antarctic minke whale', "Minkley's cichlid", 'Mink frog', 'European mink', 'american mink', 'bluemink', 'common minke whale', 'north pacific minke whale', 'antarctic minke whale', "minkley's cichlid", 'mink frog', 'european mink']
vison/mustela içeren anahtarlar: []


In [8]:
print(taxonomy.ensure_ncbi_tax_id("American mink"))
print(taxonomy.ensure_ncbi_tax_id("European mink"))

452646
9666


In [10]:
mink_rows = [rec for rec in raw_tab if rec['organism'] == 'MINK']
print(f"Toplam MINK satırı: {len(mink_rows)}")
for rec in mink_rows:
    print("-" * 40)
    print("complex_id:", rec['complex_id'])
    print("complex_name:", rec['complex_name'])
    print("subunits_organism:", rec['subunits_organism'])

Toplam MINK satırı: 4
----------------------------------------
complex_id: 321
complex_name: CCND2-CDK4-CDKN1B complex
subunits_organism: Neovison vison (American mink);Neovison vison (American mink);Neovison vison (American mink)
----------------------------------------
complex_id: 3030
complex_name: TGFbetaRII-TGFbetaRI complex
subunits_organism: Homo sapiens (Human);Homo sapiens (Human)
----------------------------------------
complex_id: 3967
complex_name: Ubiquitin E3 ligase (SMURF2, SMAD2), TGF(beta)-dependent
subunits_organism: Neovison vison (American mink);Neovison vison (American mink)
----------------------------------------
complex_id: 3971
complex_name: SMURF2-SMAD3 complex, TGF(beta)-dependent
subunits_organism: Neovison vison (American mink);Neovison vison (American mink)


In [11]:
importlib.reload(corum)

complexes_all = corum.corum_complexes(organism=None)

for cid in ['321', '3030', '3967', '3971']:
    matches = [c for c in complexes_all.values() if cid in c.ids.get('CORUM', set())]
    if matches:
        c = matches[0]
        print(f"complex_id={cid}  name={c.name}  ncbi_tax_id={c.ncbi_tax_id}")
    else:
        print(f"complex_id={cid}  -> hala atlanmis")

complex_id=321  name=CCND2-CDK4-CDKN1B complex  ncbi_tax_id=452646
complex_id=3030  name=TGFbetaRII-TGFbetaRI complex  ncbi_tax_id=9606
complex_id=3967  name=Ubiquitin E3 ligase (SMURF2, SMAD2), TGF(beta)-dependent  ncbi_tax_id=452646
complex_id=3971  name=SMURF2-SMAD3 complex, TGF(beta)-dependent  ncbi_tax_id=452646


In [12]:
print(len(complexes_all))  # önceki 8097'den 4 fazla olmalı, yani 8101 civarı (Mammalia'lar zaten dahildi)

8101


In [1]:
from pypath.utils import taxonomy
result = taxonomy.ensure_ncbi_tax_id('MINK')
print(result)

[2026-08-09 19:26:43] [curl] Module `pysftp` not available. Only downloading of a small number of resources relies on this module. Please install by PIP if it is necessary for you.
None


In [2]:
print(taxonomy.ensure_ncbi_tax_id('Neovison vison (American mink)'))

452646


In [8]:
import importlib
import pypath.inputs.corum as corum
importlib.reload(corum)

complexes_all = corum.corum_complexes(organism=None)
print(len(complexes_all))  # 8101 civarı olmalı

8101


In [7]:
import csv
import pypath.share.curl as curl
import pypath.resources.urls as urls

url = urls.urls['corum']['url']
c = curl.Curl(url, large=True, silent=False)
tab = csv.DictReader(c.result, delimiter='\t')

for rec in tab:
    if rec['complex_id'] == '340':
        print("uniprot raw:", repr(rec['subunits_uniprot_id']))
        print("genesymbol raw:", repr(rec['subunits_gene_name']))
        print("uniprot count:", len(rec['subunits_uniprot_id'].strip(';').split(';')))
        print("genesymbol count:", len(rec['subunits_gene_name'].strip(';').split(';')))
        break

uniprot raw: 'Q28646;Q28685;G1T8Y6;Q28686;Q28635;G1SGL0;Q28626;P82352'
genesymbol raw: ';DAG1;DMD;SGCA;SGCB;SGCD;SNTA1;SSPN'
uniprot count: 8
genesymbol count: 7


In [9]:
import collections

seen_count = collections.Counter()
key_to_ids = collections.defaultdict(set)

for k, c in complexes_all.items():
    key_to_ids[k] |= c.ids.get('CORUM', set())

merged = {k: ids for k, ids in key_to_ids.items() if len(ids) > 1}
print(f"Birden fazla CORUM complex_id'sinin aynı key'e düştüğü durum sayısı: {len(merged)}")

for k, ids in list(merged.items())[:5]:
    print(k, "->", ids)

Birden fazla CORUM complex_id'sinin aynı key'e düştüğü durum sayısı: 0


In [10]:
print(list(rec.keys()))

['complex_id', 'complex_name', 'synonyms', 'organism', 'cell_line', 'pmid', 'comment_complex', 'comment_members', 'comment_disease', 'comment_drug', 'comment_drug_formal', 'subunits_uniprot_id', 'subunits_gene_name', 'subunits_gene_name_synonyms', 'subunits_protein_name', 'subunits_proteinname_synonyms', 'subunits_organism', 'subunits_drugs', 'subunits_mims', 'subunits_stoechiometrie', 'purification_methods', 'functions_evi', 'functions_pmid', 'functions_go_id', 'functions_go_name', 'fcgs_id', 'fcgs_name', 'fcgs_category_name']
